In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
Firms = "/content/drive/MyDrive/India_Firms"
import pandas as pd

In [3]:
df = pd.read_parquet(f"{Firms}/India_Firms.parquet")

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc
import time
import json

# ============================================================
# FULL 2012–2026 LOCAL-CLUSTER PRODUCTION
# ============================================================

PRODUCTION_ROOT = Path(
    "/content/fire_event_production/full_2012_2026"
)

CLUSTER_DIR = PRODUCTION_ROOT / "clusters"
EDGE_DIR = PRODUCTION_ROOT / "edges"
EVENT_DIR = PRODUCTION_ROOT / "events"
LOG_DIR = PRODUCTION_ROOT / "logs"
MAPPING_DIR = PRODUCTION_ROOT / "mappings"

for directory in [
    PRODUCTION_ROOT,
    CLUSTER_DIR,
    EDGE_DIR,
    EVENT_DIR,
    LOG_DIR,
    MAPPING_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 70)
print("FULL 2012–2026 PRODUCTION DIRECTORIES")
print("=" * 70)

print("Production root:", PRODUCTION_ROOT)
print("Cluster dir    :", CLUSTER_DIR)
print("Edge dir       :", EDGE_DIR)
print("Event dir      :", EVENT_DIR)
print("Log dir        :", LOG_DIR)
print("Mapping dir    :", MAPPING_DIR)

print("\n✅ Directories ready")


FULL 2012–2026 PRODUCTION DIRECTORIES
Production root: /content/fire_event_production/full_2012_2026
Cluster dir    : /content/fire_event_production/full_2012_2026/clusters
Edge dir       : /content/fire_event_production/full_2012_2026/edges
Event dir      : /content/fire_event_production/full_2012_2026/events
Log dir        : /content/fire_event_production/full_2012_2026/logs
Mapping dir    : /content/fire_event_production/full_2012_2026/mappings

✅ Directories ready


In [6]:
# ============================================================
# COMPACT DATE INDEX
# ============================================================

date_index = {}

for date_value, group in df.groupby("date", sort=False):
    date_index[pd.Timestamp(date_value).date()] = group.index.to_numpy()

print("Indexed dates:", len(date_index))

assert len(date_index) == 5337
assert pd.Timestamp("2012-03-25").date() not in date_index

print("✅ Date index ready")


Indexed dates: 5337
✅ Date index ready


In [7]:
# ============================================================
# OBSERVED DATE LIST
# ============================================================

observed_dates = sorted(
    date_index.keys()
)

print("=" * 70)
print("OBSERVED DATE RANGE")
print("=" * 70)

print("Observed dates:", len(observed_dates))
print("First date    :", observed_dates[0])
print("Last date     :", observed_dates[-1])

calendar_dates = pd.date_range(
    min(observed_dates),
    max(observed_dates),
    freq="D"
).date

missing_dates = sorted(
    set(calendar_dates) - set(observed_dates)
)

print("Calendar dates:", len(calendar_dates))
print("Missing dates :", len(missing_dates))

print("\nMissing dates:")
for d in missing_dates:
    print(" ", d)

assert len(observed_dates) == 5337
assert observed_dates[0] == pd.Timestamp("2012-01-20").date()
assert observed_dates[-1] == pd.Timestamp("2026-08-31").date()
assert missing_dates == [
    pd.Timestamp("2012-03-25").date()
]

print("\n✅ Date coverage validation passed")


OBSERVED DATE RANGE
Observed dates: 5337
First date    : 2012-01-20
Last date     : 2026-08-31
Calendar dates: 5338
Missing dates : 1

Missing dates:
  2012-03-25

✅ Date coverage validation passed


In [8]:
# ============================================================
# PRODUCTION LOG
# ============================================================

PRODUCTION_LOG = LOG_DIR / "local_cluster_production_log.csv"

LOG_COLUMNS = [
    "date",
    "status",
    "n_detections",
    "n_spatial_pairs",
    "n_valid_pairs",
    "n_clusters",
    "output_file",
    "elapsed_seconds",
    "error"
]

if PRODUCTION_LOG.exists():

    production_log = pd.read_csv(
        PRODUCTION_LOG
    )

else:

    production_log = pd.DataFrame(
        columns=LOG_COLUMNS
    )

print("=" * 70)
print("LOCAL-CLUSTER PRODUCTION LOG")
print("=" * 70)

print("Log file:", PRODUCTION_LOG)
print("Existing records:", len(production_log))

if len(production_log):
    print(
        "Completed dates:",
        (
            production_log["status"]
            .eq("COMPLETED")
            .sum()
        )
    )

print("✅ Production log ready")


LOCAL-CLUSTER PRODUCTION LOG
Log file: /content/fire_event_production/full_2012_2026/logs/local_cluster_production_log.csv
Existing records: 0
✅ Production log ready


In [9]:
# ============================================================
# LOG WRITER
# ============================================================

def save_production_log(log_df):
    """
    Save production progress atomically.
    """

    tmp_path = PRODUCTION_LOG.with_suffix(
        ".tmp.csv"
    )

    log_df.to_csv(
        tmp_path,
        index=False
    )

    tmp_path.replace(
        PRODUCTION_LOG
    )


In [10]:
# ============================================================
# FULL LOCAL-CLUSTER PRODUCTION RUNNER
# ============================================================

def run_full_local_cluster_production(
    observed_dates,
    date_index,
    df,
    cluster_dir,
    production_log,
    stop_after=None
):
    """
    Full daily local-cluster production.

    Each observed date is processed independently.

    Output:
        clusters_YYYY-MM-DD.parquet

    The missing calendar date 2012-03-25 is naturally skipped
    because it is absent from observed_dates.
    """

    print("=" * 70)
    print("STARTING FULL 2012–2026 LOCAL-CLUSTER PRODUCTION")
    print("=" * 70)

    total_dates = len(observed_dates)

    completed_dates = set(
        production_log.loc[
            production_log["status"] == "COMPLETED",
            "date"
        ].astype(str)
    )

    print("Total observed dates :", total_dates)
    print("Already completed    :", len(completed_dates))
    print("Remaining             :", total_dates - len(completed_dates))

    run_counter = 0

    for date_value in observed_dates:

        date_string = pd.Timestamp(
            date_value
        ).strftime("%Y-%m-%d")

        output_path = (
            cluster_dir /
            f"clusters_{date_string}.parquet"
        )

        # ----------------------------------------------------
        # Skip validated existing output
        # ----------------------------------------------------

        if (
            date_string in completed_dates
            and output_path.exists()
        ):

            continue

        # ----------------------------------------------------
        # Remove stale log entry for this date
        # ----------------------------------------------------

        production_log = production_log[
            production_log["date"].astype(str)
            != date_string
        ].copy()

        # ----------------------------------------------------
        # Extract daily detections
        # ----------------------------------------------------

        indices = date_index.get(
            pd.Timestamp(date_value).date()
        )

        if indices is None or len(indices) == 0:

            print(
                f"⚠️ No detections for {date_string}"
            )

            continue

        day_df = df.loc[
            indices
        ]

        start = time.time()

        print("\n" + "-" * 70)
        print(
            f"PROCESSING {date_string}"
        )
        print("-" * 70)

        print(
            "Detections:",
            len(day_df)
        )

        try:

            # ------------------------------------------------
            # VALIDATED DAILY CLUSTERING FUNCTION
            # ------------------------------------------------

            work_day, summary, diagnostics = (
                cluster_single_day(
                    day_df
                )
            )

            # ------------------------------------------------
            # Add date explicitly
            # ------------------------------------------------

            summary.insert(
                0,
                "date",
                date_string
            )

            # ------------------------------------------------
            # Write cluster summary
            # ------------------------------------------------

            summary.to_parquet(
                output_path,
                index=False
            )

            elapsed = time.time() - start

            # ------------------------------------------------
            # Record successful completion
            # ------------------------------------------------

            record = {
                "date": date_string,
                "status": "COMPLETED",
                "n_detections": diagnostics[
                    "n_detections"
                ],
                "n_spatial_pairs": diagnostics[
                    "n_spatial_pairs"
                ],
                "n_valid_pairs": diagnostics[
                    "n_valid_pairs"
                ],
                "n_clusters": diagnostics[
                    "n_clusters"
                ],
                "output_file": str(
                    output_path
                ),
                "elapsed_seconds": elapsed,
                "error": ""
            }

            production_log = pd.concat(
                [
                    production_log,
                    pd.DataFrame([record])
                ],
                ignore_index=True
            )

            save_production_log(
                production_log
            )

            print(
                "Spatial pairs:",
                diagnostics["n_spatial_pairs"]
            )

            print(
                "Valid pairs:",
                diagnostics["n_valid_pairs"]
            )

            print(
                "Local clusters:",
                diagnostics["n_clusters"]
            )

            print(
                f"Elapsed: {elapsed:.2f} s"
            )

            print(
                "✅ SAVED:",
                output_path.name
            )

            # ------------------------------------------------
            # Release daily objects
            # ------------------------------------------------

            del work_day
            del summary
            del day_df
            gc.collect()

            run_counter += 1

            # ------------------------------------------------
            # Optional controlled test
            # ------------------------------------------------

            if (
                stop_after is not None
                and run_counter >= stop_after
            ):

                print("\n")
                print("=" * 70)
                print(
                    "CONTROLLED STOP REACHED"
                )
                print("=" * 70)

                break

        except Exception as e:

            elapsed = time.time() - start

            record = {
                "date": date_string,
                "status": "FAILED",
                "n_detections": len(day_df),
                "n_spatial_pairs": np.nan,
                "n_valid_pairs": np.nan,
                "n_clusters": np.nan,
                "output_file": str(
                    output_path
                ),
                "elapsed_seconds": elapsed,
                "error": repr(e)
            }

            production_log = pd.concat(
                [
                    production_log,
                    pd.DataFrame([record])
                ],
                ignore_index=True
            )

            save_production_log(
                production_log
            )

            print(
                f"❌ FAILED: {date_string}"
            )

            print(
                "Error:",
                repr(e)
            )

            # Stop rather than silently continuing.
            raise

    print("\n")
    print("=" * 70)
    print("LOCAL-CLUSTER PRODUCTION RUN FINISHED")
    print("=" * 70)

    completed = (
        production_log["status"]
        .eq("COMPLETED")
        .sum()
    )

    failed = (
        production_log["status"]
        .eq("FAILED")
        .sum()
    )

    print("Completed:", completed)
    print("Failed   :", failed)

    return production_log


In [13]:
# ============================================================
# RESTORE VALIDATED DAILY CLUSTERING FUNCTION
# ============================================================

import numpy as np
import pandas as pd

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components


def cluster_single_day(
    day_df,
    spatial_threshold_m=SPATIAL_THRESHOLD_M,
    temporal_threshold_seconds=TEMPORAL_THRESHOLD_SECONDS
):
    """
    Validated daily fire-event clustering.

    Direct edge:
        distance <= 1,000 m
        AND
        |time difference| <= 12 h

    Local clusters:
        connected components of the valid-edge graph.
    """

    if day_df.empty:
        return pd.DataFrame(), pd.DataFrame(), {
            "n_detections": 0,
            "n_spatial_pairs": 0,
            "n_valid_pairs": 0,
            "n_clusters": 0
        }

    work = day_df.copy()

    required = [
        "datetime",
        "x",
        "y",
        "frp",
        "latitude",
        "longitude",
        "satellite"
    ]

    missing = [
        c for c in required
        if c not in work.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}"
        )

    work = work.reset_index(
        drop=False
    ).rename(
        columns={"index": "_original_index"}
    )

    n = len(work)

    # --------------------------------------------------------
    # Spatial search
    # --------------------------------------------------------

    xy = work[
        ["x", "y"]
    ].to_numpy(dtype=np.float64)

    tree = cKDTree(xy)

    if n > 1:

        pairs = tree.query_pairs(
            r=spatial_threshold_m,
            output_type="ndarray"
        )

        pairs = np.asarray(
            pairs,
            dtype=np.int64
        ).reshape(-1, 2)

    else:

        pairs = np.empty(
            (0, 2),
            dtype=np.int64
        )

    n_spatial_pairs = len(pairs)

    # --------------------------------------------------------
    # Temporal filtering
    # --------------------------------------------------------

    if n_spatial_pairs > 0:

        times = (
            work["datetime"]
            .astype("int64")
            .to_numpy()
        )

        dt_seconds = np.abs(
            times[pairs[:, 0]]
            -
            times[pairs[:, 1]]
        ) / 1_000_000_000.0

        valid_pairs = pairs[
            dt_seconds <= temporal_threshold_seconds
        ]

    else:

        valid_pairs = np.empty(
            (0, 2),
            dtype=np.int64
        )

    n_valid_pairs = len(valid_pairs)

    # --------------------------------------------------------
    # Connected components
    # --------------------------------------------------------

    if n == 1:

        labels = np.zeros(
            1,
            dtype=np.int64
        )

    elif n_valid_pairs == 0:

        labels = np.arange(
            n,
            dtype=np.int64
        )

    else:

        rows = np.concatenate([
            valid_pairs[:, 0],
            valid_pairs[:, 1]
        ])

        cols = np.concatenate([
            valid_pairs[:, 1],
            valid_pairs[:, 0]
        ])

        data = np.ones(
            len(rows),
            dtype=np.uint8
        )

        graph = coo_matrix(
            (
                data,
                (rows, cols)
            ),
            shape=(n, n)
        ).tocsr()

        _, labels = connected_components(
            graph,
            directed=False,
            return_labels=True
        )

    # --------------------------------------------------------
    # Globally unique local cluster ID
    # --------------------------------------------------------

    day_value = pd.Timestamp(
        work["datetime"].min()
    ).strftime("%Y%m%d")

    work["local_cluster_id"] = [
        f"{day_value}_{int(label)}"
        for label in labels
    ]

    # --------------------------------------------------------
    # Cluster statistics
    # --------------------------------------------------------

    summary_rows = []

    for cluster_id, g in work.groupby(
        "local_cluster_id",
        sort=False
    ):

        start_time = g["datetime"].min()
        end_time = g["datetime"].max()

        cx = g["x"].mean()
        cy = g["y"].mean()

        dx = (
            g["x"].to_numpy(dtype=np.float64)
            - cx
        )

        dy = (
            g["y"].to_numpy(dtype=np.float64)
            - cy
        )

        max_radius_m = np.sqrt(
            dx * dx + dy * dy
        ).max()

        satellites = sorted(
            g["satellite"]
            .astype(str)
            .unique()
            .tolist()
        )

        summary_rows.append({
            "local_cluster_id": cluster_id,
            "start_time": start_time,
            "end_time": end_time,
            "n_detections": len(g),
            "centroid_x": cx,
            "centroid_y": cy,
            "total_frp": g["frp"].sum(),
            "mean_frp": g["frp"].mean(),
            "max_frp": g["frp"].max(),
            "n_observation_times": g["datetime"].nunique(),
            "duration_hours": (
                end_time - start_time
            ).total_seconds() / 3600.0,
            "satellites": ",".join(satellites),
            "n_satellites": len(satellites),
            "max_radius_m": max_radius_m,
            "max_radius_km": max_radius_m / 1000.0
        })

    summary = pd.DataFrame(
        summary_rows
    )

    # --------------------------------------------------------
    # Projected centroid -> WGS84
    # --------------------------------------------------------

    import geopandas as gpd
    from shapely.geometry import Point

    points = gpd.GeoSeries(
        [
            Point(x, y)
            for x, y in zip(
                summary["centroid_x"],
                summary["centroid_y"]
            )
        ],
        crs="EPSG:6933"
    )

    geographic = points.to_crs(
        "EPSG:4326"
    )

    summary["centroid_lon"] = (
        geographic.x.to_numpy()
    )

    summary["centroid_lat"] = (
        geographic.y.to_numpy()
    )

    diagnostics = {
        "n_detections": n,
        "n_spatial_pairs": n_spatial_pairs,
        "n_valid_pairs": n_valid_pairs,
        "n_clusters": len(summary)
    }

    return work, summary, diagnostics


print("✅ cluster_single_day restored")


NameError: name 'SPATIAL_THRESHOLD_M' is not defined

In [12]:
# ============================================================
# SMOKE TEST OUTPUT QC
# ============================================================

smoke_files = sorted(
    CLUSTER_DIR.glob("clusters_*.parquet")
)

print("=" * 70)
print("SMOKE TEST OUTPUT QC")
print("=" * 70)

print("Cluster files:", len(smoke_files))

for p in smoke_files:
    print(
        p.name,
        f"{p.stat().st_size / 1024**2:.3f} MB"
    )

assert len(smoke_files) == 10

print("\n✅ 10 cluster files created")


SMOKE TEST OUTPUT QC
Cluster files: 0


AssertionError: 

In [14]:
# ============================================================
# FRESH NOTEBOOK — LOAD COMBINED VIIRS DATA
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import time

DATA_PATH = Path(
    "/content/drive/MyDrive/India_Firms/India_Firms.parquet"
)

print("=" * 70)
print("LOADING COMBINED VIIRS DATA")
print("=" * 70)

print("Input:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

assert DATA_PATH.exists(), (
    f"Input file not found: {DATA_PATH}"
)

df = pd.read_parquet(DATA_PATH)

print("\nRows   :", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())


LOADING COMBINED VIIRS DATA
Input: /content/drive/MyDrive/India_Firms/India_Firms.parquet
Exists: True

Rows   : 15392720
Columns: 22

Columns:
['latitude', 'longitude', 'brightness', 'scan', 'track', 'acq_date', 'acq_time', 'satellite', 'instrument', 'confidence', 'version', 'bright_t31', 'frp', 'daynight', 'datetime', 'date', 'x', 'y', 'grid_x', 'grid_y', 'time_seconds', 'time_bin']


In [15]:
# ============================================================
# PRODUCTION CONSTANTS
# ============================================================

SPATIAL_THRESHOLD_M = 1000.0

TEMPORAL_THRESHOLD_HOURS = 12.0

TEMPORAL_THRESHOLD_SECONDS = (
    TEMPORAL_THRESHOLD_HOURS * 3600.0
)

print("=" * 70)
print("PRODUCTION PARAMETERS")
print("=" * 70)

print(
    "Spatial threshold:",
    SPATIAL_THRESHOLD_M,
    "m"
)

print(
    "Temporal threshold:",
    TEMPORAL_THRESHOLD_HOURS,
    "hours"
)

print(
    "Temporal threshold:",
    TEMPORAL_THRESHOLD_SECONDS,
    "seconds"
)


PRODUCTION PARAMETERS
Spatial threshold: 1000.0 m
Temporal threshold: 12.0 hours
Temporal threshold: 43200.0 seconds


In [16]:
# ============================================================
# DATETIME PREPARATION
# ============================================================

print("=" * 70)
print("DATETIME CHECK")
print("=" * 70)

print(
    "datetime exists:",
    "datetime" in df.columns
)

print(
    "date exists:",
    "date" in df.columns
)


DATETIME CHECK
datetime exists: True
date exists: True


In [17]:
# ============================================================
# DATETIME VALIDATION
# ============================================================

if "datetime" in df.columns:

    df["datetime"] = pd.to_datetime(
        df["datetime"],
        errors="coerce"
    )

else:

    required = [
        "acq_date",
        "acq_time"
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    assert not missing, (
        f"Missing datetime fields: {missing}"
    )

    # FIRMS acq_time is HHMM
    df["acq_time_str"] = (
        df["acq_time"]
        .astype(str)
        .str.zfill(4)
    )

    df["datetime"] = pd.to_datetime(
        df["acq_date"].astype(str)
        + " "
        + df["acq_time_str"],
        format="%Y-%m-%d %H%M",
        errors="coerce"
    )

    df.drop(
        columns=["acq_time_str"],
        inplace=True
    )

# ------------------------------------------------------------
# Date field
# ------------------------------------------------------------

df["date"] = (
    df["datetime"]
    .dt.normalize()
)

print(
    "Bad datetime:",
    df["datetime"].isna().sum()
)

print(
    "First date:",
    df["date"].min()
)

print(
    "Last date:",
    df["date"].max()
)

assert df["datetime"].notna().all()

print("\n✅ Datetime ready")


Bad datetime: 0
First date: 2012-01-20 00:00:00
Last date: 2026-08-31 00:00:00

✅ Datetime ready


In [ ]:
# ============================================================
# REBUILD EPSG:6933 COORDINATES
# ============================================================

required_coordinates = [
    "latitude",
    "longitude"
]

missing = [
    c for c in required_coordinates
    if c not in df.columns
]

assert not missing, (
    f"Missing coordinate columns: {missing}"
)

print("=" * 70)
print("COORDINATE PREPARATION")
print("=" * 70)

print(
    "Bad latitude:",
    (
        (~df["latitude"].between(-90, 90))
        | df["latitude"].isna()
    ).sum()
)

print(
    "Bad longitude:",
    (
        (~df["longitude"].between(-180, 180))
        | df["longitude"].isna()
    ).sum()
)

assert df["latitude"].between(
    -90, 90
).all()

assert df["longitude"].between(
    -180, 180
).all()

import geopandas as gpd

geometry = gpd.points_from_xy(
    df["longitude"],
    df["latitude"]
)

gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

print("\nConverting EPSG:4326 → EPSG:6933")

gdf = gdf.to_crs(
    "EPSG:6933"
)

df["x"] = gdf.geometry.x.to_numpy(
    dtype=np.float64
)

df["y"] = gdf.geometry.y.to_numpy(
    dtype=np.float64
)

del gdf
del geometry

gc.collect()

print("\nX range:")
print(
    df["x"].min(),
    "→",
    df["x"].max()
)

print("\nY range:")
print(
    df["y"].min(),
    "→",
    df["y"].max()
)

print("\n✅ EPSG:6933 coordinates rebuilt")


COORDINATE PREPARATION
Bad latitude: 0
Bad longitude: 0


In [ ]:
# ============================================================
# SOURCE DATA QC
# ============================================================

print("=" * 70)
print("SOURCE DATA QC")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", len(df.columns))

bad_datetime = df["datetime"].isna().sum()

bad_latitude = (
    (~df["latitude"].between(-90, 90))
    | df["latitude"].isna()
).sum()

bad_longitude = (
    (~df["longitude"].between(-180, 180))
    | df["longitude"].isna()
).sum()

bad_x = df["x"].isna().sum()
bad_y = df["y"].isna().sum()

print("Bad datetime :", bad_datetime)
print("Bad latitude :", bad_latitude)
print("Bad longitude:", bad_longitude)
print("Bad x        :", bad_x)
print("Bad y        :", bad_y)

assert len(df) == 15392720
assert bad_datetime == 0
assert bad_latitude == 0
assert bad_longitude == 0
assert bad_x == 0
assert bad_y == 0

print("\n✅ SOURCE DATA QC PASSED")
